<a href="https://colab.research.google.com/github/kjahan/duplicate_detection/blob/main/notebooks/BigBird.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigBirdModel

https://arxiv.org/pdf/2007.14062.pdf

https://huggingface.co/docs/transformers/model_doc/big_bird

In [32]:
from transformers import AutoTokenizer, BigBirdModel
import torch

from numpy import dot
from numpy.linalg import norm
import numpy as np

## Error wo sentencepiece

ValueError: Couldn't instantiate the backend tokenizer from one of:
(1) a `tokenizers` library serialization file,
(2) a slow tokenizer instance to convert or
(3) an equivalent slow tokenizer class to instantiate and convert.
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.


Solution

`Uninstalled transformers`

`Installed transformers sentencepiece like this : !pip install --no-cache-dir transformers sentencepiece`

`Use_fast= False like this: tokenizer = AutoTokenizer.from_pretrained(“XXXXX”, use_fast=False)`

https://discuss.huggingface.co/t/error-with-new-tokenizers-urgent/2847/3



In [11]:
!pip uninstall transformers --y

Found existing installation: transformers 4.35.2
Uninstalling transformers-4.35.2:
  Successfully uninstalled transformers-4.35.2


In [12]:
!pip uninstall sentencepiece --y

Found existing installation: sentencepiece 0.1.99
Uninstalling sentencepiece-0.1.99:
  Successfully uninstalled sentencepiece-0.1.99


In [13]:
!pip install --no-cache-dir transformers sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 23.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00


In [2]:
tokenizer = AutoTokenizer.from_pretrained("google/bigbird-roberta-base", use_fast=False)

In [3]:
model = BigBirdModel.from_pretrained("google/bigbird-roberta-base")

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

In [36]:
inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

In [5]:
outputs = model(**inputs)

Attention type 'block_sparse' is not possible if sequence_length: 8 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


In [6]:
last_hidden_states = outputs.last_hidden_state

In [10]:
v1 = last_hidden_states

In [11]:
v1.shape

torch.Size([1, 8, 768])

In [12]:
arr_1 = v1.cpu().detach().numpy()

In [13]:
arr_1.shape

(1, 8, 768)

In [19]:
arr_1[0][0].shape

(768,)

In [20]:
v11 = arr_1[0][0]

## Vector 2

In [37]:
inputs = tokenizer("Hello, my dog is funny", return_tensors="pt")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state
v = last_hidden_states
arr = v.cpu().detach().numpy()
v21 = arr[0][0]

In [38]:
v11 = v11.reshape(-1,1)
v21 = v21.reshape(-1,1)

v11 = np.squeeze(np.asarray(v11))
v21 = np.squeeze(np.asarray(v21))

In [39]:
cos_sim = dot(v11, v21)/(norm(v11)*norm(v21))

In [40]:
cos_sim

0.97565264

## Let's use this a doc similairty

In [52]:
d1 = "Hello, my dog is cute"
d2 = "Hello, my dog is funny"

# which hidden state vector to read and compare there are 8 of them
for inx in range(8):
  i = inx
  inputs = tokenizer(d1, return_tensors="pt")
  outputs = model(**inputs)
  last_hidden_states = outputs.last_hidden_state
  v = last_hidden_states
  arr = v.cpu().detach().numpy()
  v11 = arr[0][i]

  # which hidden state vector to read and compare there are 8 of them
  j = inx
  inputs = tokenizer(d2, return_tensors="pt")
  outputs = model(**inputs)
  last_hidden_states = outputs.last_hidden_state
  v = last_hidden_states
  arr = v.cpu().detach().numpy()
  v21 = arr[0][j]

  # v11 = v11.reshape(-1,1)
  # v21 = v21.reshape(-1,1)

  v11 = np.squeeze(np.asarray(v11))
  v21 = np.squeeze(np.asarray(v21))

  cos_sim = dot(v11, v21)/(norm(v11)*norm(v21))

  print("hidden state inx: {} --> cos_sim: {}\n\n".format(inx, cos_sim))

hidden state inx: 0 --> cos_sim: 0.9756526350975037


hidden state inx: 1 --> cos_sim: 0.9792460799217224


hidden state inx: 2 --> cos_sim: 0.9817030429840088


hidden state inx: 3 --> cos_sim: 0.9800967574119568


hidden state inx: 4 --> cos_sim: 0.9855754375457764


hidden state inx: 5 --> cos_sim: 0.9728330373764038


hidden state inx: 6 --> cos_sim: 0.9007318019866943


hidden state inx: 7 --> cos_sim: 0.9950714707374573




## Long doc

https://en.wikipedia.org/wiki/Jaguar_Cars

https://en.wikipedia.org/wiki/Jaguar

In [47]:
d1 = """Jaguar (UK: /ˈdʒæɡjuər/, US: /ˈdʒæɡwɑːr/) is the luxury vehicle brand of Jaguar Land Rover,[1][3] a British multinational car manufacturer with its headquarters in Whitley, Coventry, England. Jaguar Cars was the company that was responsible for the production of Jaguar cars until its operations were fully merged with those of Land Rover to form Jaguar Land Rover on 1 January 2013.

Jaguar's business was founded as the Swallow Sidecar Company in 1922, originally making motorcycle sidecars before developing bodies for passenger cars. Under the ownership of SS Cars, the business extended to complete cars made in association with Standard Motor Company, many bearing Jaguar as a model name. The company's name was changed from SS Cars to Jaguar Cars in 1945. A merger with the British Motor Corporation followed in 1966,[4] the resulting enlarged company now being renamed as British Motor Holdings (BMH), which in 1968 merged with Leyland Motor Corporation and became British Leyland, itself to be nationalised in 1975.

Jaguar was spun off from British Leyland and was listed on the London Stock Exchange in 1984 until it was acquired by Ford in 1990.[5] Since the late 1970s, Jaguar manufactured cars for the Prime Minister of the United Kingdom,[6][7][8] the most recent prime ministerial car delivery being an XJ (X351) in May 2010.[9][10][11] The company also held royal warrants from Queen Elizabeth II and Prince Charles.[12]

Ford owned Jaguar Cars, also buying Land Rover in 2000, until 2008 when it sold both to Tata Motors. Tata created Jaguar Land Rover as a subsidiary holding company. At operating company level, Jaguar Cars was merged in 2013 with Land Rover to form Jaguar Land Rover as the single design, manufacture, sales company, and brand owner for both Jaguar and Land Rover vehicles.

Since the Ford ownership era, Jaguar and Land Rover have used joint design facilities in engineering centres at Whitley in Coventry and Gaydon in Warwickshire and Jaguar cars have been assembled in plants at Castle Bromwich and Solihull. On 15 February 2021, Jaguar Land Rover announced that all cars made under the Jaguar brand will be fully electric by 2025.[13]"""

In [45]:
d2 = """The jaguar (Panthera onca) is a large cat species and the only living member of the genus Panthera native to the Americas. With a body length of up to 1.85 m (6 ft 1 in) and a weight of up to 158 kg (348 lb), it is the biggest cat species in the Americas and the third largest in the world. Its distinctively marked coat features pale yellow to tan colored fur covered by spots that transition to rosettes on the sides, although a melanistic black coat appears in some individuals. The jaguar's powerful bite allows it to pierce the carapaces of turtles and tortoises, and to employ an unusual killing method: it bites directly through the skull of mammalian prey between the ears to deliver a fatal blow to the brain.

The modern jaguar's ancestors probably entered the Americas from Eurasia during the Early Pleistocene via the land bridge that once spanned the Bering Strait. Today, the jaguar's range extends from the Southwestern United States across Mexico and much of Central America, the Amazon rainforest and south to Paraguay and northern Argentina. It inhabits a variety of forested and open terrains, but its preferred habitat is tropical and subtropical moist broadleaf forest, wetlands and wooded regions. It is adept at swimming and is largely a solitary, opportunistic, stalk-and-ambush apex predator. As a keystone species, it plays an important role in stabilizing ecosystems and in regulating prey populations.

The jaguar is threatened by habitat loss, habitat fragmentation, poaching for trade with its body parts and killings in human–wildlife conflict situations, particularly with ranchers in Central and South America. It has been listed as Near Threatened on the IUCN Red List since 2002. The wild population is thought to have declined since the late 1990s. Priority areas for jaguar conservation comprise 51 Jaguar Conservation Units (JCUs), defined as large areas inhabited by at least 50 breeding jaguars. The JCUs are located in 36 geographic regions ranging from Mexico to Argentina.

The jaguar has featured prominently in the mythology of indigenous peoples of the Americas, including those of the Aztec and Maya civilizations."""

In [51]:
# which hidden state vector to read and compare there are 8 of them
for inx in range(8):
  i = inx
  inputs = tokenizer(d1, return_tensors="pt")
  outputs = model(**inputs)
  last_hidden_states = outputs.last_hidden_state
  v = last_hidden_states
  arr = v.cpu().detach().numpy()
  v11 = arr[0][i]

  # which hidden state vector to read and compare there are 8 of them
  j = inx
  inputs = tokenizer(d2, return_tensors="pt")
  outputs = model(**inputs)
  last_hidden_states = outputs.last_hidden_state
  v = last_hidden_states
  arr = v.cpu().detach().numpy()
  v21 = arr[0][j]

  # v11 = v11.reshape(-1,1)
  # v21 = v21.reshape(-1,1)

  v11 = np.squeeze(np.asarray(v11))
  v21 = np.squeeze(np.asarray(v21))

  cos_sim = dot(v11, v21)/(norm(v11)*norm(v21))

  print("hidden state inx: {} --> cos_sim: {}\n\n".format(inx, cos_sim))

hidden state inx: 0 --> cos_sim: 0.9735521078109741


hidden state inx: 1 --> cos_sim: 0.5652112364768982


hidden state inx: 2 --> cos_sim: 0.5596461296081543


hidden state inx: 3 --> cos_sim: 0.19302096962928772


hidden state inx: 4 --> cos_sim: 0.4657732844352722


hidden state inx: 5 --> cos_sim: 0.562473475933075


hidden state inx: 6 --> cos_sim: 0.40464577078819275


hidden state inx: 7 --> cos_sim: 0.4889627695083618


